# Robot Diagnostics — Full Hardware Test Suite

**Purpose:** Run every subsystem independently to identify what's working and what's broken.

**How to use:**
1. Run each numbered section in order.
2. Each section is self-contained — you can re-run it without restarting.
3. Read the markdown before running a section — it tells you what to expect and any safety warnings.
4. Copy the **Final Summary** output and paste it into your next request for fixes.

**Test order:**
| # | Section | Hardware? | Risk |
|---|---------|-----------|------|
| 1 | Setup | No | None |
| 2 | State Machine Logic | No | None |
| 3 | Camera | Yes (CSI) | None |
| 4 | Motors | Yes (tracks) | **Robot moves** |
| 5 | Servos / Arm | Yes (arm) | **Arm moves** |
| 6 | Perception | Yes (camera) | None |
| 7 | Integration | Yes (all) | **Robot moves** |
| 8 | Summary | No | None |

---
## 1. Setup

Initializes the test result accumulator.

**Run this first** every time you start or restart the notebook.

In [ ]:
import sys
import os

# Auto-detect project root by searching for config.yaml marker
project_root = os.getcwd()
while not os.path.exists(os.path.join(project_root, 'config.yaml')) and project_root != '/':
    project_root = os.path.dirname(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from robot_tests import test_results
test_results.reset()

print('=' * 50)
print('Robot Diagnostics — Test Suite Ready')
print('=' * 50)
print()
print('Next: run Section 2 (State Machine Logic).')

Robot Diagnostics — Test Suite Ready

Next: run Section 2 (State Machine Logic).


---
## 2. State Machine Logic Tests

**What it tests:** Navigation state transitions, turn-direction logic, avoidance timing, speed clamping.

**Hardware required:** None — these are pure Python logic tests.

**Expected output:** All tests should PASS. If any FAIL, there's a bug in the navigation algorithm itself.

**Tests:**
- `WANDER` → `AVOID_BOUNDARY` when yellow detected
- `WANDER` → `AVOID_OBSTACLE` when obstacle detected
- Boundary takes priority over obstacle
- Turn direction: away from the side with more signal
- Avoidance timing phases (reverse → turn → wander)
- Cooldown period blocks new detection briefly
- Speed clamp keeps motor values within limits
- `STOPPED` state blocks all transitions

In [2]:
from robot_tests.test_state_machine import run_all as run_state_machine

print('Running state machine logic tests...')
print()
run_state_machine()

print('Done. If all passed, your navigation logic is solid.')
print('Next: Section 3 (Camera).')

Running state machine logic tests...

--- State Machine Tests (no hardware) ---

Done. If all passed, your navigation logic is solid.
Next: Section 3 (Camera).


---
## 3. Camera Tests

**What it tests:** Camera initialization, frame capture, resolution, format, and OpenCV fallback.

**Hardware required:** CSI camera connected to Jetson Nano.

**Safety:** None — camera only, no movement.

**Common failures:**
- `Could not initialize camera` → Another notebook or process is using the camera. Fix: `Kernel → Shutdown All Kernels`, then re-run.
- `Fallback Camera ready` → Jetbot wrapper failed but OpenCV works. This is fine — perception will still work.
- `camera.value is None` → Camera initialized but no frames arriving. Check cable connection.

**Tests:**
- Camera init (jetbot wrapper)
- Camera init (OpenCV fallback)
- Frame read (valid BGR image)
- Frame format (3 channels)
- Resolution matches config (300×300)

In [3]:
from robot_tests.test_camera import run_all as run_camera

print('Running camera tests...')
print()
camera, source = run_camera()

print()
if source == 'jetbot':
    print('Camera: JetBot Camera (native wrapper)')
elif source == 'fallback_ok':
    print('Camera: OpenCV fallback (works fine for perception)')
else:
    print('Camera: FAILED — no camera available')
    print()
    print('TROUBLESHOOTING:')
    print('1. Check camera cable is seated in the CSI port')
    print('2. Run: sudo pkill -f gst-launch')
    print('3. Kernel → Shutdown All Kernels → re-open notebook')

print()
print('Next: Section 4 (Motors) — robot WILL move, keep clear.')

Running camera tests...

--- Camera Tests ---


Camera: JetBot Camera (native wrapper)

Next: Section 4 (Motors) — robot WILL move, keep clear.


---
## 4. Motor Tests

**What it tests:** Robot initialization, motor directions, stop, speed limits.

**Hardware required:** Robot chassis with motor HAT.

**SAFETY WARNING:** The robot will drive forward, reverse, and spin during these tests. **Place it on blocks or keep a clear area of at least 1 metre around it.**

**Common failures:**
- `Robot init failed` → I2C not configured or motor HAT not detected. Run `sudo i2cdetect -y 1` and look for `0x60` or `0x70`.
- `Motor forward FAIL` → One or both motors not responding. Check motor wiring to HAT.
- Motors spin wrong direction → Swap the two wires on that motor's terminal block.

**Tests:**
- Robot init (creates Robot object)
- Motor stop (verifies both motors at 0.0)
- Forward drive (0.5 s at speed 0.15)
- Reverse drive (0.5 s at speed 0.15)
- Turn left (0.5 s spin)
- Turn right (0.5 s spin)
- Speed clamp (verifies max speed 0.25 accepted)

In [4]:
from robot_tests.test_motors import run_all as run_motors

print('!' * 50)
print('WARNING: Robot will move. Keep clear or place on blocks.')
print('!' * 50)
print()

# Adjust duration and speed if needed
DURATION = 0.5   # seconds each direction
SPEED    = 0.15  # motor speed (0.0 - 1.0)
MAX_SPEED = 0.25

print(f'Duration: {DURATION}s per direction, Speed: {SPEED}')
print()

robot = run_motors(duration=DURATION, speed=SPEED, max_speed=MAX_SPEED)

print()
if robot is not None:
    print('Motors OK. Next: Section 5 (Servos) — arm WILL move, keep clear.')
else:
    print('Motors FAILED. Check I2C and wiring before proceeding.')

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Duration: 0.5s per direction, Speed: 0.15

--- Motor Tests ---



Motors OK. Next: Section 5 (Servos) — arm WILL move, keep clear.


---
## 5. Servo / Arm Tests

**What it tests:** TTLServo library, camera pan/tilt, arm joints (base, shoulder, elbow), gripper open/close, home position.

**Hardware required:** Servo controller board connected to Jetson Nano via ttyTHS1.

**SAFETY WARNING:** The robot arm will move through its range of motion. **Remove any fragile objects nearby and keep children away.**

**Common failures:**
- `Servo library not available` → `SCSCtrl` not installed or `ttyTHS1` permission denied. Fix: `sudo chmod 777 /dev/ttyTHS1` then restart kernel.
- Servo doesn't move but library loads → Check servo power supply (must be 6–7.4V).
- Servo jitters or overheats → ID conflict. Verify each servo has a unique ID (1–6).

**Tests:**
- Servo library import
- Camera pan (servo 1) and tilt (servo 5)
- Arm base (servo 2)
- Arm shoulder (servo 3)
- Arm elbow (servo 4)
- Gripper open/close (servo 6)
- Return all servos to home position (0°)

In [5]:
from robot_tests.test_servos import run_all as run_servos

print('!' * 50)
print('WARNING: Arm will move. Keep clear of fragile objects.')
print('!' * 50)
print()

servo = run_servos()

print()
if servo is not None:
    print('Servos OK. Next: Section 6 (Perception) — needs working camera.')
else:
    print('Servos FAILED. Check ttyTHS1 permissions and servo power.')

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

--- Servo Tests ---

Succeeded to open the port
Succeeded to change the baudrate


Servos OK. Next: Section 6 (Perception) — needs working camera.


---
## 6. Perception Tests

**What it tests:** Yellow tape detection, obstacle edge detection, and frame capture rate. These are the core vision algorithms used during autonomous navigation.

**Hardware required:** Working camera (Section 3 must have passed).

**Safety:** None — camera only, no movement.

**How to interpret results:**
- **Yellow detection:** Shows pixel count. Hold yellow tape in front of camera → count should jump above threshold.
- **Obstacle detection:** Shows edge pixel count. Hold a box in front of camera → count should jump above threshold.
- **Frame rate:** Should be > 10 fps for real-time navigation. If < 5 fps, the Jetson may be throttled (check power supply).

**Tests:**
- Frame available from camera
- Yellow tape detection (pixel count in bottom ROI)
- Obstacle edge detection (Canny edges in front ROI)
- Frame rate measurement

In [6]:
from robot_tests.test_perception import run_all as run_perception

print('Running perception tests...')
print()

# Use camera from Section 3 if available, otherwise try fallback
try:
    _ = camera
except NameError:
    camera = None
    source = 'fallback'

run_perception(camera=camera, source=source if 'source' in dir() else 'jetbot')

print()
print('Next: Section 7 (Integration) — robot may move briefly.')

Running perception tests...

--- Perception Tests ---


Next: Section 7 (Integration) — robot may move briefly.


---
## 7. Integration Tests

**What it tests:** End-to-end pipelines: camera → perception, emergency stop sequence, reset logic, safe_stop with missing hardware.

**Hardware required:** Camera (for pipeline test), robot (for stop test).

**Safety:** Robot may briefly move during safe_stop test. Keep clear.

**Tests:**
- `safe_stop` with `robot = None` (must not crash)
- `safe_stop` with real robot (must zero motors)
- Camera → yellow detection → obstacle detection pipeline
- Emergency stop sequence (unobserve + stop + state=STOPPED)
- Reset to WANDER (all globals cleared)

In [7]:
from robot_tests.test_integration import run_all as run_integration

print('Running integration tests...')
print()
run_integration()

print()
print('Next: Section 8 (Summary) — copy the output for your report.')

Running integration tests...

--- Integration Tests ---


Next: Section 8 (Summary) — copy the output for your report.


---
## 8. Final Summary

Prints a consolidated pass/fail report of every test you ran.

**Copy and paste this output** into your next request so I can generate a v2 fix for everything that failed.

In [8]:
print()
print('=' * 60)
print('FINAL TEST SUMMARY')
print('=' * 60)
print()

results = test_results.summary()

print()
print('Copy everything above this line and paste it into your next request.')
print('I will generate a v2 with fixes for all failed tests.')


FINAL TEST SUMMARY

  TEST SUMMARY
  Total:  38
  Passed: 38
  Failed: 0
  Time:   8.28s
------------------------------------------------------------
  [OK] PASS  — State WANDER->AVOID_BOUNDARY
         transitioned on yellow detect
  [OK] PASS  — State WANDER->AVOID_OBSTACLE
         transitioned on obstacle detect
  [OK] PASS  — Boundary priority
         boundary wins over obstacle
  [OK] PASS  — Turn direction (boundary)
         more yellow left -> turn right
  [OK] PASS  — Turn direction (obstacle)
         more edges left -> turn right
  [OK] PASS  — Avoid timing (phase 1)
         reverse phase active
  [OK] PASS  — Avoid timing (done)
         avoidance complete -> WANDER
  [OK] PASS  — Cooldown active
         cooldown for 0.5s
  [OK] PASS  — Cooldown expired
         detection re-enabled
  [OK] PASS  — Speed clamp logic
         all cases correct
  [OK] PASS  — STOPPED state
         blocks all transitions
  [OK] PASS  — Camera init (jetbot)
         resolution 320x240
  [O

---
## Appendix A: Re-Test Individual Modules

If a specific subsystem fails and you want to re-test it after fixing, run the corresponding cell below. You don't need to re-run the whole suite.

In [9]:
# Re-test: Camera only
from robot_tests.test_camera import run_all as run_camera
camera, source = run_camera()

--- Camera Tests ---



In [10]:
# Re-test: Motors only (robot will move)
from robot_tests.test_motors import run_all as run_motors
robot = run_motors(duration=0.5, speed=0.15, max_speed=0.25)

--- Motor Tests ---




In [11]:
# Re-test: Servos only (arm will move)
from robot_tests.test_servos import run_all as run_servos
servo = run_servos()

--- Servo Tests ---




In [12]:
# Re-test: Perception only (needs camera)
from robot_tests.test_perception import run_all as run_perception
run_perception(camera=camera if 'camera' in dir() else None,
               source=source if 'source' in dir() else 'jetbot')

--- Perception Tests ---



In [13]:
# Re-test: State machine only (no hardware)
from robot_tests.test_state_machine import run_all as run_state_machine
run_state_machine()

--- State Machine Tests (no hardware) ---



---
## Appendix B: Quick Troubleshooting

| Symptom | Likely Cause | Quick Fix |
|---------|------------|-----------|
| `Could not initialize camera` | Camera locked by another process | `Kernel → Shutdown All Kernels`, reopen notebook |
| `Permission denied: /dev/ttyTHS1` | Serial port not accessible | Terminal: `sudo chmod 777 /dev/ttyTHS1` |
| `Robot init failed` | I2C/motor HAT not detected | Terminal: `sudo i2cdetect -y 1`, look for `0x60` |
| Motor spins wrong way | Wires reversed | Swap the two wires on that motor's terminal |
| Servo doesn't move | No power or wrong ID | Check 6V supply, verify servo IDs 1–6 |
| Yellow detection always 0 | HSV range wrong | Adjust `TAPE_HSV_LOWER/UPPER` in test |
| Frame rate < 5 fps | Jetson under-powered | Use 5V 4A power supply, not USB |